# Phase 2: Chapman Chemistry Verification

Verifies MPAS + MICM Chapman chemistry. Key checks:
- O3 shows a diurnal cycle (production on dayside, none on nightside)
- Single-column values match MUSICA tutorial 10
- Mass conservation for O + O1D + O3

**Pre-requisite:** `data/jw_480km/output.nc`

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("..") / "data" / "jw_480km"
OUTPUT = DATA_DIR / "output.nc"
assert OUTPUT.exists(), f"Run the Phase 2 test first — {OUTPUT} not found"

ds = nc.Dataset(OUTPUT)
lat = np.degrees(ds["latCell"][:])
lon = np.degrees(ds["lonCell"][:])
nTimes = ds.dimensions["Time"].size
print(f"Grid: {ds.dimensions['nCells'].size} cells, {nTimes} time steps")

## 1. O3 Diurnal Cycle

Map of ozone at a mid-level at two different times of day.
Dayside should show O3 production/destruction; nightside should be static.

In [ ]:
o3_name = "o3"
if o3_name in ds.variables:
    lev = 12  # ~135 hPa, stratospheric
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, t, label in zip(axes, [0, nTimes//2], ["t=0 (midnight)", f"t={nTimes//2} (noon)"]):
        vals = ds[o3_name][t, :, lev]
        sc = ax.scatter(lon, lat, c=vals, s=4, cmap="YlGn")
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.set_title(f"O3 at ~135 hPa — {label}")
        plt.colorbar(sc, ax=ax, label="mol/mol")
    plt.tight_layout()
    plt.show()
else:
    print(f"Variable '{o3_name}' not found — Phase 2 not yet implemented")
    print(f"Available variables: {list(ds.variables.keys())}")

## 2. O3 Time Series

Domain-mean O3 at a stratospheric level over the simulation.
Should show a clear diurnal signal.

In [ ]:
if o3_name in ds.variables:
    lev = 12
    o3_mean = np.array([ds[o3_name][t, :, lev].mean() for t in range(nTimes)])
    o3_max = np.array([ds[o3_name][t, :, lev].max() for t in range(nTimes)])
    o3_min = np.array([ds[o3_name][t, :, lev].min() for t in range(nTimes)])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(range(nTimes), o3_min, o3_max, alpha=0.2, color="tab:green")
    ax.plot(range(nTimes), o3_mean, "o-", markersize=3, label="mean")
    ax.set_xlabel("Time step (hours)")
    ax.set_ylabel("O3 (mol/mol)")
    ax.set_title("O3 at ~135 hPa — Domain Statistics")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Odd Oxygen Conservation (O + O1D + O3)

Total odd oxygen should be conserved by Chapman chemistry.

In [ ]:
ox_species = ["o", "o1d", "o3"]
found = [s for s in ox_species if s in ds.variables]
if len(found) == 3:
    ox_total = np.array([
        sum(ds[s][t, :, :].sum() for s in found)
        for t in range(nTimes)
    ])
    # Use late-time reference for relative change (species start at zero)
    ref = ox_total[-1] if ox_total[-1] != 0 else 1.0
    rel_change = (ox_total - ox_total[-1]) / ref
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(nTimes), rel_change, "o-", markersize=3)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Relative Ox change (vs final)")
    ax.set_title("Odd Oxygen (O + O1D + O3) — Relative to Final Time")
    ax.axhline(0, color="k", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
    # Check late-time stability (last 5 steps)
    if nTimes > 5:
        late_ox = ox_total[-5:]
        late_var = (late_ox.max() - late_ox.min()) / late_ox.mean() if late_ox.mean() != 0 else 0
        print(f"Late-time Ox variability (last 5 steps): {late_var:.2e}")
    print(f"Ox at t=0: {ox_total[0]:.4e}, Ox at t=-1: {ox_total[-1]:.4e}")
else:
    print(f"Found {found} of {ox_species} — Phase 2 not yet complete")

ds.close()